In [1]:
"""
External evaluation of the trained M2 (Ben Graham + CBAM) model on the
Roboflow generalization dataset. INFERENCE ONLY — no training happens here.

Model class, preprocessing, and class order are copied EXACTLY from your
training notebook (m2_bengram.ipynb) and training_config.json.

Metrics reported: BOTH "simple" (overall/weighted) and macro-averaged
accuracy, precision, recall, F1 — plus full per-class breakdown.

Visualizations saved (matching your training artifact style):
  - confusion_matrix.png            raw-count heatmap
  - confusion_matrix_normalized.png row-normalized heatmap (% per true class)
  - per_class_accuracy.png          bar chart, recall per class + support
  - per_class_metrics_bar.png       grouped bar: precision/recall/F1 per class
  - roc_curves.png                  one-vs-rest ROC + AUC per class
  - overall_metrics_summary.png     simple vs macro metrics side-by-side
  - misclassified_grid.png          montage of wrong predictions (true/pred)
"""

import csv
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
    roc_curve,
    auc,
)
from sklearn.preprocessing import label_binarize

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
EXTERNAL_DATA_DIR = r"C:\Users\R Y Z E N\Desktop\roboflow_bengram_illumination"
MODEL_PATH        = r"C:\Users\R Y Z E N\Desktop\m2_bengram\best_model.pth"
OUTPUT_DIR        = r"C:\Users\R Y Z E N\Desktop\m2_bengram\external_eval_roboflow"

IMG_SIZE = 224
BATCH_SIZE = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

CLASS_NAMES = [
    "Calculus",
    "Caries",
    "Gingivitis",
    "Hypodontia",
    "Mouth_Ulcer",
    "Tooth_Discoloration",
]

CBAM_REDUCTION = 16
CBAM_SPATIAL_KERNEL = 7
DROPOUT = 0.3
MAX_MISCLASSIFIED_SHOWN = 16


# ---------------------------------------------------------------------------
# Model — copied exactly from training notebook
# ---------------------------------------------------------------------------
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(in_planes, max(in_planes // reduction, 1), 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(max(in_planes // reduction, 1), in_planes, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(x_cat))


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(spatial_kernel)

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x


class EfficientNetCBAM(nn.Module):
    def __init__(self, num_classes, cbam_reduction=16, cbam_kernel=7, dropout=0.3, pretrained=False):
        super().__init__()
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        base = models.efficientnet_b0(weights=weights)
        self.features = base.features
        self.cbam = CBAM(1280, cbam_reduction, cbam_kernel)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(1280, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.cbam(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


# ---------------------------------------------------------------------------
# Dataset — images already Ben-Graham-preprocessed; just resize/tensor/normalize
# ---------------------------------------------------------------------------
class ExternalEvalDataset(Dataset):
    def __init__(self, root_dir: str, class_names: list, transform=None):
        self.root_dir = Path(root_dir)
        self.class_names = class_names
        self.class_to_idx = {name: i for i, name in enumerate(class_names)}
        self.transform = transform
        self.samples = []

        valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
        for class_name in class_names:
            class_dir = self.root_dir / class_name
            if not class_dir.exists():
                print(f"WARNING: expected class folder not found: {class_dir}")
                continue
            for img_path in class_dir.iterdir():
                if img_path.suffix.lower() in valid_exts:
                    self.samples.append((img_path, self.class_to_idx[class_name]))

        if not self.samples:
            raise RuntimeError(f"No images found under {root_dir} for classes {class_names}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label, str(img_path)


# ---------------------------------------------------------------------------
# Model loading
# ---------------------------------------------------------------------------
def load_model(model_path: str, num_classes: int, device: torch.device) -> nn.Module:
    model = EfficientNetCBAM(
        num_classes=num_classes,
        cbam_reduction=CBAM_REDUCTION,
        cbam_kernel=CBAM_SPATIAL_KERNEL,
        dropout=DROPOUT,
        pretrained=False,
    )
    state_dict = torch.load(model_path, map_location=device)

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing or unexpected:
        print(f"[WARNING] Missing keys: {missing}")
        print(f"[WARNING] Unexpected keys: {unexpected}")
        print(">>> Architecture should match exactly — investigate before trusting results.")
    else:
        print("State dict loaded cleanly — architecture matches best_model.pth exactly.")

    model.to(device)
    model.eval()
    return model


# ---------------------------------------------------------------------------
# Visualizations
# ---------------------------------------------------------------------------
def plot_confusion_matrix(cm_arr, class_names, output_dir):
    plt.figure(figsize=(8, 7))
    sns.heatmap(cm_arr, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title("Confusion Matrix — External Roboflow Eval")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.savefig(output_dir / "confusion_matrix.png", dpi=150)
    plt.close()

    # normalized version — % of each true class, easier to compare against
    # classes with very different support counts
    row_sums = cm_arr.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm_arr, row_sums, out=np.zeros_like(cm_arr, dtype=float), where=row_sums != 0)
    plt.figure(figsize=(8, 7))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, vmin=0, vmax=1)
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title("Confusion Matrix (Row-Normalized) — External Roboflow Eval")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.savefig(output_dir / "confusion_matrix_normalized.png", dpi=150)
    plt.close()


def plot_per_class_accuracy(cm_arr, class_names, output_dir):
    row_sums = cm_arr.sum(axis=1)
    per_class_acc = np.divide(np.diag(cm_arr), row_sums,
                               out=np.zeros_like(row_sums, dtype=float), where=row_sums != 0)
    plt.figure(figsize=(8, 5))
    bars = plt.bar(class_names, per_class_acc, color="steelblue")
    for bar, acc, support in zip(bars, per_class_acc, row_sums):
        plt.text(bar.get_x() + bar.get_width() / 2, acc + 0.01,
                  f"{acc:.2f}\n(n={int(support)})", ha="center", fontsize=8)
    plt.ylim(0, 1.15)
    plt.ylabel("Per-class Accuracy (Recall)")
    plt.title("Per-class Accuracy — External Roboflow Eval")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.savefig(output_dir / "per_class_accuracy.png", dpi=150)
    plt.close()


def plot_per_class_metrics_bar(all_labels, all_preds, class_names, output_dir):
    precisions, recalls, f1s, supports = precision_recall_fscore_support(
        all_labels, all_preds, labels=list(range(len(class_names))), zero_division=0
    )
    x = np.arange(len(class_names))
    width = 0.25

    plt.figure(figsize=(10, 5.5))
    plt.bar(x - width, precisions, width, label="Precision", color="#4C72B0")
    plt.bar(x, recalls, width, label="Recall", color="#DD8452")
    plt.bar(x + width, f1s, width, label="F1", color="#55A868")
    plt.xticks(x, class_names, rotation=25, ha="right")
    plt.ylim(0, 1.15)
    plt.ylabel("Score")
    plt.title("Per-class Precision / Recall / F1 — External Roboflow Eval")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_dir / "per_class_metrics_bar.png", dpi=150)
    plt.close()


def plot_roc_curves(all_labels, all_probs, class_names, output_dir):
    num_classes = len(class_names)
    y_true_bin = label_binarize(all_labels, classes=list(range(num_classes)))
    y_probs = np.array(all_probs)

    plt.figure(figsize=(8, 7))
    for i in range(num_classes):
        # skip classes with no positive examples in this external set —
        # ROC/AUC is undefined without at least one positive sample
        if y_true_bin[:, i].sum() == 0:
            print(f"  Skipping ROC for {class_names[i]}: no positive samples in external set")
            continue
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{class_names[i]} (AUC={roc_auc:.3f})")
    plt.plot([0, 1], [0, 1], "k--", linewidth=1)
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC Curves (One-vs-Rest) — External Roboflow Eval")
    plt.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.savefig(output_dir / "roc_curves.png", dpi=150)
    plt.close()


def plot_overall_metrics_summary(simple_acc, weighted_p, weighted_r, weighted_f1,
                                  macro_p, macro_r, macro_f1, output_dir):
    labels = ["Accuracy", "Precision", "Recall", "F1"]
    simple_vals = [simple_acc, weighted_p, weighted_r, weighted_f1]
    macro_vals = [simple_acc, macro_p, macro_r, macro_f1]  # accuracy has no macro variant

    x = np.arange(len(labels))
    width = 0.35

    plt.figure(figsize=(7, 5))
    bars1 = plt.bar(x - width / 2, simple_vals, width, label="Simple / Weighted", color="#4C72B0")
    bars2 = plt.bar(x + width / 2, macro_vals, width, label="Macro", color="#DD8452")
    for bars in (bars1, bars2):
        for bar in bars:
            h = bar.get_height()
            plt.text(bar.get_x() + bar.get_width() / 2, h + 0.01, f"{h:.3f}",
                      ha="center", fontsize=8)
    plt.xticks(x, labels)
    plt.ylim(0, 1.15)
    plt.ylabel("Score")
    plt.title("Overall Metrics — Simple vs Macro — External Roboflow Eval")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_dir / "overall_metrics_summary.png", dpi=150)
    plt.close()


def plot_misclassified_grid(all_paths, all_labels, all_preds, class_names,
                             img_size, output_dir, max_samples=16):
    wrong = [(p, t, pr) for p, t, pr in zip(all_paths, all_labels, all_preds) if t != pr]
    if not wrong:
        print("No misclassified external samples — skipping misclassified grid.")
        return
    wrong = wrong[:max_samples]
    n = len(wrong)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.6, rows * 2.8))
    axes = np.array(axes).reshape(-1)
    for i, (path, true_idx, pred_idx) in enumerate(wrong):
        img = Image.open(path).convert("RGB").resize((img_size, img_size))
        axes[i].imshow(img)
        axes[i].set_title(f"true: {class_names[true_idx]}\npred: {class_names[pred_idx]}",
                          fontsize=8, color="red")
        axes[i].axis("off")
    for j in range(len(wrong), len(axes)):
        axes[j].axis("off")
    plt.suptitle(f"Misclassified External Samples ({len(wrong)} shown)", fontsize=11)
    plt.tight_layout()
    plt.savefig(output_dir / "misclassified_grid.png", dpi=130)
    plt.close()


# ---------------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------------
def evaluate(model_path: str, data_dir: str, output_dir: str):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    eval_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ])

    dataset = ExternalEvalDataset(data_dir, CLASS_NAMES, transform=eval_transform)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    print(f"Loaded {len(dataset)} external images across {len(CLASS_NAMES)} classes.")
    model = load_model(model_path, num_classes=len(CLASS_NAMES), device=DEVICE)

    all_preds, all_labels, all_paths, all_probs = [], [], [], []

    with torch.no_grad():
        for imgs, labels, paths in loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = torch.argmax(outputs, dim=1).cpu().numpy()

            all_preds.extend(preds.tolist())
            all_labels.extend(labels.numpy().tolist())
            all_paths.extend(paths)
            all_probs.extend(probs.tolist())

    all_labels_arr = np.array(all_labels)
    all_preds_arr = np.array(all_preds)

    # ---- Metrics ----
    simple_acc = accuracy_score(all_labels_arr, all_preds_arr)
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        all_labels_arr, all_preds_arr, average="weighted", zero_division=0
    )
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        all_labels_arr, all_preds_arr, average="macro", zero_division=0
    )
    report = classification_report(
        all_labels_arr, all_preds_arr, target_names=CLASS_NAMES, digits=4, zero_division=0
    )
    cm = confusion_matrix(all_labels_arr, all_preds_arr, labels=list(range(len(CLASS_NAMES))))

    # ---- Print ----
    print("\n" + "=" * 70)
    print(f"EXTERNAL EVALUATION — {data_dir}")
    print(f"Total images: {len(dataset)}")
    print("=" * 70)
    print("\n-- SIMPLE / OVERALL METRICS --")
    print(f"Accuracy:            {simple_acc:.4f}")
    print(f"Weighted Precision:  {weighted_p:.4f}")
    print(f"Weighted Recall:     {weighted_r:.4f}")
    print(f"Weighted F1:         {weighted_f1:.4f}")
    print("\n-- MACRO METRICS (each class weighted equally) --")
    print(f"Macro Precision:     {macro_p:.4f}")
    print(f"Macro Recall:        {macro_r:.4f}")
    print(f"Macro F1:            {macro_f1:.4f}")
    print("\n-- PER-CLASS REPORT --")
    print(report)
    print("Confusion matrix (rows=true, cols=pred):")
    print(CLASS_NAMES)
    print(cm)

    # ---- Visualizations ----
    print("\nGenerating visualizations...")
    plot_confusion_matrix(cm, CLASS_NAMES, output_dir)
    plot_per_class_accuracy(cm, CLASS_NAMES, output_dir)
    plot_per_class_metrics_bar(all_labels_arr, all_preds_arr, CLASS_NAMES, output_dir)
    plot_roc_curves(all_labels_arr, all_probs, CLASS_NAMES, output_dir)
    plot_overall_metrics_summary(simple_acc, weighted_p, weighted_r, weighted_f1,
                                  macro_p, macro_r, macro_f1, output_dir)
    plot_misclassified_grid(all_paths, all_labels_arr, all_preds_arr, CLASS_NAMES,
                             IMG_SIZE, output_dir, max_samples=MAX_MISCLASSIFIED_SHOWN)
    print("Visualizations saved: confusion_matrix.png, confusion_matrix_normalized.png,")
    print("  per_class_accuracy.png, per_class_metrics_bar.png, roc_curves.png,")
    print("  overall_metrics_summary.png, misclassified_grid.png")

    # ---- Save text/csv outputs ----
    summary_lines = [
        f"EXTERNAL EVALUATION — {data_dir}",
        f"Model: {model_path}",
        f"Total images: {len(dataset)}",
        "=" * 70,
        "",
        "SIMPLE / OVERALL METRICS",
        f"  Accuracy:            {simple_acc:.4f}",
        f"  Weighted Precision:  {weighted_p:.4f}",
        f"  Weighted Recall:     {weighted_r:.4f}",
        f"  Weighted F1:         {weighted_f1:.4f}",
        "",
        "MACRO METRICS (each class weighted equally)",
        f"  Macro Precision:     {macro_p:.4f}",
        f"  Macro Recall:        {macro_r:.4f}",
        f"  Macro F1:            {macro_f1:.4f}",
        "",
        "PER-CLASS REPORT",
        report,
        "",
        "CONFUSION MATRIX (rows=true, cols=pred)",
        str(CLASS_NAMES),
        np.array2string(cm),
    ]
    with open(output_dir / "external_eval_summary.txt", "w") as f:
        f.write("\n".join(summary_lines))

    with open(output_dir / "confusion_matrix.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([""] + CLASS_NAMES)
        for i, row in enumerate(cm):
            writer.writerow([CLASS_NAMES[i]] + row.tolist())

    with open(output_dir / "per_image_predictions.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["image_path", "true_class", "predicted_class"])
        for path, label, pred in zip(all_paths, all_labels_arr, all_preds_arr):
            writer.writerow([path, CLASS_NAMES[label], CLASS_NAMES[pred]])

    print(f"\nAll results and visualizations written to: {output_dir}")


if __name__ == "__main__":
    evaluate(MODEL_PATH, EXTERNAL_DATA_DIR, OUTPUT_DIR)

Loaded 1057 external images across 6 classes.


C:\Users\R Y Z E N\AppData\Local\Temp\ipykernel_33636\3286498067.py:185: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location=devic

State dict loaded cleanly — architecture matches best_model.pth exactly.

EXTERNAL EVALUATION — C:\Users\R Y Z E N\Desktop\roboflow_bengram_illumination
Total images: 1057

-- SIMPLE / OVERALL METRICS --
Accuracy:            0.9953
Weighted Precision:  0.9953
Weighted Recall:     0.9953
Weighted F1:         0.9953

-- MACRO METRICS (each class weighted equally) --
Macro Precision:     0.9945
Macro Recall:        0.9945
Macro F1:            0.9945

-- PER-CLASS REPORT --
                     precision    recall  f1-score   support

           Calculus     0.9818    0.9818    0.9818       110
             Caries     0.9901    1.0000    0.9950       200
         Gingivitis     1.0000    1.0000    1.0000       147
         Hypodontia     1.0000    0.9899    0.9949       199
        Mouth_Ulcer     0.9950    1.0000    0.9975       200
Tooth_Discoloration     1.0000    0.9950    0.9975       201

           accuracy                         0.9953      1057
          macro avg     0.9945    0

In [2]:
import hashlib
from pathlib import Path
from collections import defaultdict

def hash_file(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()

def build_hash_index(root_dir):
    index = defaultdict(list)
    for p in Path(root_dir).rglob("*"):
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:
            index[hash_file(p)].append(str(p))
    return index

train_hashes = build_hash_index(r"C:\Users\R Y Z E N\Desktop\ben_gram_illumination_dataset")
roboflow_hashes = build_hash_index(r"C:\Users\R Y Z E N\Desktop\roboflow_dataset_merged")

overlap = set(train_hashes) & set(roboflow_hashes)
print(f"Exact duplicate images: {len(overlap)}")
for h in list(overlap)[:10]:
    print(train_hashes[h], "<->", roboflow_hashes[h])

Exact duplicate images: 0


In [ ]:
from PIL import Image
import imagehash
from pathlib import Path

def build_phash_index(root_dir):
    index = {}
    for p in Path(root_dir).rglob("*"):
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:
            try:
                h = imagehash.phash(Image.open(p))
                index[str(p)] = h
            except Exception as e:
                print(f"Skipped {p}: {e}")
    return index

print("Hashing training set...")
train_p = build_phash_index(r"C:\Users\R Y Z E N\Desktop\ben_gram_illumination_dataset")
print(f"  {len(train_p)} images hashed")

print("Hashing Roboflow set...")
rob_p = build_phash_index(r"C:\Users\R Y Z E N\Desktop\roboflow_dataset_merged")
print(f"  {len(rob_p)} images hashed")

# Threshold guide: 0 = identical, <=5 = near-identical (likely same source
# image, different compression/resize), <=10 = visually similar (could be
# same tooth/photo session, or could just be coincidence for simple images)
THRESHOLD = 5

near_dupes = []
train_items = list(train_p.items())
for i, (rpath, rhash) in enumerate(rob_p.items()):
    if i % 100 == 0:
        print(f"  Checked {i}/{len(rob_p)} Roboflow images...")
    for tpath, thash in train_items:
        dist = rhash - thash
        if dist <= THRESHOLD:
            near_dupes.append((rpath, tpath, dist))

near_dupes.sort(key=lambda x: x[2])  # closest matches first
print(f"\nNear-duplicate pairs found (threshold={THRESHOLD}): {len(near_dupes)}")
for rpath, tpath, dist in near_dupes[:20]:
    print(f"  dist={dist}  {rpath}  <->  {tpath}")

Hashing training set...
  15866 images hashed
Hashing Roboflow set...
  1057 images hashed
  Checked 0/1057 Roboflow images...
  Checked 100/1057 Roboflow images...
  Checked 200/1057 Roboflow images...
  Checked 300/1057 Roboflow images...
  Checked 400/1057 Roboflow images...
  Checked 500/1057 Roboflow images...
  Checked 600/1057 Roboflow images...
  Checked 700/1057 Roboflow images...
  Checked 800/1057 Roboflow images...


In [ ]:
import matplotlib.pyplot as plt
for rpath, tpath, dist in near_dupes[:5]:
    fig, axes = plt.subplots(1, 2, figsize=(6,3))
    axes[0].imshow(Image.open(rpath)); axes[0].set_title(f"Roboflow\n{Path(rpath).name}")
    axes[1].imshow(Image.open(tpath)); axes[1].set_title(f"Train\n{Path(tpath).name}")
    for ax in axes: ax.axis("off")
    plt.suptitle(f"Hamming dist: {dist}")
    plt.show()

In [5]:
pip install ImageHash


   ---------------------------------------- 0.0/4.2 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.2 MB 4.8 MB/s eta 0:00:01
   ----------------- ---------------------- 1.8/4.2 MB 5.0 MB/s eta 0:00:01
   ----------------------------- ---------- 3.1/4.2 MB 5.1 MB/s eta 0:00:01
   ---------------------------------------  4.2/4.2 MB 5.1 MB/s eta 0:00:01
   ---------------------------------------- 4.2/4.2 MB 4.8 MB/s  0:00:00

   ---------------------------------------- 0/2 [PyWavelets]
   ---------------------------------------- 0/2 [PyWavelets]
   ---------------------------------------- 0/2 [PyWavelets]
   ---------------------------------------- 0/2 [PyWavelets]
   ---------------------------------------- 0/2 [PyWavelets]
   ---------------------------------------- 0/2 [PyWavelets]
   -------------------- ------------------- 1/2 [ImageHash]
   ---------------------------------------- 2/2 [ImageHash]

